# Congressional Bills Data Cleaning

This notebook creates the `bills.csv` data included in this repo. The code is based on the [replication code](https://osf.io/gjt87/) from Egami et al. (2023).

In [1]:
import os
import pandas as pd
import numpy as np

# Define directories
repo_dir = "/Users/haya1/Documents/LanguageModel_Labels-main/congressional_bills"
data_dir = os.path.join(repo_dir, "01_cleaning_and_data")
os.makedirs(data_dir, exist_ok=True)
os.chdir(repo_dir)

## Load data from CAP

In [2]:
data_link = "https://comparativeagendas.s3.amazonaws.com/datasetfiles/US-Legislative-congressional_bills_19.3_3_3.csv"
cap = pd.read_csv(data_link, low_memory=False)
print(f"Downloaded CAP data, n = {len(cap)}")

# Correct bill IDs
cap["bill_id_corrected"] = cap["cong"].astype(str) + "-" + cap["bill_type"].apply(lambda x: x.upper()) + "-" + cap["bill_no"].astype(str) 
cap_wrong_bill_id = cap[cap["bill_id"] != cap["bill_id_corrected"]]
print(f"Corrected bill_id of {len(cap_wrong_bill_id)} observations")

# Remove duplicates based on bill ID
cap_duplicates = cap[cap["bill_id_corrected"].duplicated(keep=False)]
n_pre = len(cap)
cap.drop_duplicates(subset=["bill_id_corrected"], inplace=True)
print(f"Dropped {n_pre - len(cap)} duplicates, n = {len(cap)}")

Downloaded CAP data, n = 468437
Corrected bill_id of 13361 observations
Dropped 4 duplicates, n = 468433


In [3]:
# Drop unnecessary columns
n_pre = len(cap.columns)
cap.drop(columns=[
"id", "bill_id", "cong", "bill_type", "chamber", "bill_no", "name_full",
"intr_date", "intr_month", "pap_majortopic", "pap_subtopic", "subtopic",
"filter_plaw", "plaw_date", "plaw_no", "majority", "comment"
], inplace=True)
print(f"Dropped {n_pre - len(cap.columns)} columns")

# Rename columns for consistancy
cap.rename(columns={
    'bill_id_corrected': 'BillID_corrected',
    'majortopic': 'Major',
    'description': 'Description',
    'year': 'Year',
    'party': 'Party', 
    'pass_h': 'PassH', 
    'pass_s': 'PassS'}, inplace=True)

# Rearrange columns
cap = cap[['BillID_corrected', 'Description', 'Year', 'Major',  'Party', 'PassH', 'PassS']]

# Display the first few rows of CAP data
cap.head()

Dropped 17 columns


,BillID_corrected,Description,Year,Major,Party,PassH,PassS
0,80-S-1873,"A bill to maintain prosperity, promote full em...",1947,1.0,200.0,0.0,0.0
1,80-HR-3000,To amend the Employment Act of1946 so as to pr...,1947,1.0,200.0,0.0,0.0
2,80-HR-4743,"To maintain prosperity, to promote full employ...",1947,1.0,200.0,0.0,0.0
3,80-S-525,A bill to promote the progress of science and ...,1947,1.0,100.0,0.0,0.0
4,80-HR-1815,To promote the progress of science; to advance...,1947,1.0,200.0,0.0,0.0


## Load covariates from CBP

In [4]:
cbp_80_92 = pd.read_csv("http://congressionalbills.org/billfiles/bills80-92.zip", sep='\t', encoding='latin-1', low_memory=False)
cbp_80_92.rename(columns={' ': 'Minor'}, inplace=True)
cbp_80_92.drop(columns=['id', 'ByReq', 'Commem', 'oldMajor', 'oldMinor', 'Month',  'ReferArr', 'Year', 'Age', 'ComMArr', 'DW2', 'LeadCham', 'LeadComm', 'LeadSubC'], inplace=True)
cbp_80_92["DW1"] = cbp_80_92["DW1"].apply(lambda x: np.nan if x == -99.0 else x)

cbp_93_114 = pd.read_csv("http://congressionalbills.org/billfiles/bills93-114.zip", sep=';', encoding='latin-1', low_memory=False)
cbp_93_114.drop(columns='ImpBill', inplace=True)
cbp_93_114 = cbp_93_114[cbp_80_92.columns.to_list()] # align columns, not needed but personal preference
cbp_93_114["DW1"] = cbp_93_114["DW1"].apply(lambda s: float(s.replace(",", ".") if isinstance(s, str) else s))

# Combine CBP datasets
cbp = pd.concat([cbp_80_92, cbp_93_114]).reset_index()
cbp.drop(columns='index', inplace=True)
print(f"Loaded CBP data, n={len(cbp)}")

Loaded CBP data, n=523841


In [5]:
# Correct BillID
cbp["BillID_corrected"] = cbp["Cong"].astype(str) + "-" + cbp["BillType"].apply(lambda x: x.upper()) + "-" + cbp["BillNum"].astype(str) 
cbp_wrong_bill_id = cbp[cbp["BillID"] != cbp["BillID_corrected"]]
print(f"Corrected bill_id of {len(cbp_wrong_bill_id)} observations")
cbp.drop(columns="BillID", inplace=True)

# Remove duplicates based on bill ID
cbp_duplicates = cbp[cbp[["BillID_corrected"]].duplicated(keep=False)]

for bill_id in cbp_duplicates["BillID_corrected"].unique():
    cap_major = cap.loc[cap["BillID_corrected"]==bill_id,"Major"].item()
    n_pre = len(cbp)
    cbp = cbp[~((cbp["BillID_corrected"]==bill_id) & (cbp["Major"]!=cap_major))]
    print(f"Dropped {n_pre-len(cbp)} duplicates of {bill_id} with major topic that is inconsistant with that in CAP, {cap_major}")

cbp_duplicates = cbp[cbp[["BillID_corrected"]].duplicated(keep=False)]
n_pre = len(cbp)
cbp.drop_duplicates(subset=["BillID_corrected", "Major", "Minor"], inplace=True)
print(f"Dropped {n_pre - len(cbp)} dublicates based on bill_id, keeping first one if they vary in more than one variable, n = {len(cbp)}")

Corrected bill_id of 1 observations
Dropped 2 duplicates of 107-HR-5715 with major topic that is inconsistant with that in CAP, 12.0
Dropped 1 duplicates of 107-S-3051 with major topic that is inconsistant with that in CAP, 9.0
Dropped 1 duplicates of 107-S-3160 with major topic that is inconsistant with that in CAP, 15.0
Dropped 1 dublicates based on bill_id, keeping first one if they vary in more than one variable, n = 523836


In [6]:
# Impute missing DW1 scores
cbp["DW1_impute"] = cbp["DW1"]
cbp.fillna(value={"DW1_impute": cbp["DW1"].mean()}, inplace=True)
print(f"Impute the {np.isnan(cbp['DW1']).sum()} missing DW1 scores using the mean, {cbp['DW1'].mean()}")

Impute the 7161 missing DW1 scores using the mean, -0.07263037915904587


In [7]:
# Rename columns
cbp.rename(columns={'Title': 'Description'}, inplace=True)

# Rearrange columns
cbp_first_col = ['BillID_corrected', 'Description', 'Major',  'Party', 'PassH', 'PassS']
cbp_remaining_col = [ col for col in cbp.columns if (col not in cbp_first_col) ]
cbp = cbp[cbp_first_col + cbp_remaining_col]

# Display the first few rows of CBP data
cbp.head()

,BillID_corrected,Description,Major,Party,PassH,PassS,BillNum,BillType,Chamber,Cong,...,NameLast,PooleID,Postal,State,URL,ChRef,RankRef,SubChRef,SubRankRef,DW1_impute
0,80-HR-1,To reduce individual income tax payments,1.0,200.0,1,1,1,HR,0,80,...,Knutson,5352.0,MN,33.0,http://beta.congress.gov/bill/80th-congress/ho...,NaN,NaN,NaN,NaN,0.533
1,80-HR-2,To amend the Armed Forces Leave Act of 1946 by...,16.0,200.0,0,0,2,HR,0,80,...,Van Zandt,9645.0,PA,14.0,http://beta.congress.gov/bill/80th-congress/ho...,NaN,NaN,NaN,NaN,0.224
2,80-HR-3,To amend the Armed Forces Leave Act of 1946 to...,16.0,100.0,0,0,3,HR,0,80,...,Rogers,8023.0,FL,43.0,http://beta.congress.gov/bill/80th-congress/ho...,NaN,NaN,NaN,NaN,0.003
3,80-HR-4,To safeguard the admission of evidence in cert...,12.0,100.0,1,0,4,HR,0,80,...,Hobbs,4471.0,AL,41.0,http://beta.congress.gov/bill/80th-congress/ho...,NaN,NaN,NaN,NaN,-0.125
4,80-HR-5,To protect honorably discharged veterans in th...,16.0,100.0,0,0,5,HR,0,80,...,Rankin,7731.0,MS,46.0,http://beta.congress.gov/bill/80th-congress/ho...,NaN,NaN,NaN,NaN,0.226


## Merge CAP and CBP

In [8]:
# Merge CAP and CBP datasets
df = cap.merge(cbp, left_on="BillID_corrected", right_on="BillID_corrected", suffixes=("_CAP", "_CBP"))
print(f"Merged CAP and CBP data, n = {len(df)}")

Merged CAP and CBP data, n = 468433


In [9]:
# Drop observations with missing data in key columns
missing_both = (
    df["Major_CAP"].isnull() | # As in Egami, use Major id from CAP rather than CBP
    df[['Party_CAP', 'Party_CBP']].isnull().all(axis=1) |
    df[['PassH_CAP', 'PassH_CBP']].isnull().all(axis=1) |
    df[['PassS_CAP', 'PassS_CBP']].isnull().all(axis=1)
)
df = df[~missing_both]

# df.drop(columns='Major_CBP', inplace=True)

print(f"Dropped {missing_both.sum()} observations with missing data from Party, PassH, PassS from both CAP and CBP.")

Dropped 8071 observations with missing data from Party, PassH, PassS from both CAP and CBP.


In [10]:
# Lowercase descriptions and remove duplicates baesd on set of variables
df["Description_CAP_lower"] = df["Description_CAP"].apply(lambda x: x.lower())
# df_dup_col = [ col for col in df.columns if (col not in ['BillID_corrected', 'Description_CAP', 'Description_CBP', 'BillNum', 'Major_CBP', 'URL']) ]
df_dup_col = ['Major_CAP', 'Party_CAP', 'Party_CBP', 'PassH_CAP', 'PassH_CBP', 'PassS_CAP',
       'PassS_CBP', 'DW1', 'DW1_impute', 'Year', 'BillType',
       'Chamber', 'Cong', 'Minor', 'MultNo', 'PLaw', 'ReportH', 'ReportS',
       'Veto', 'ComC', 'CumHServ', 'CumSServ', 'Gender', 'Majority',
       'MemberID', 'MRef', 'NameFirst', 'NameFull', 'NameLast', 'PooleID',
       'Postal', 'State', 'Description_CAP_lower']
df_duplicates = df[df[df_dup_col].duplicated(keep=False)]

n_pre = len(df)
df.drop_duplicates(subset=df_dup_col, inplace=True)
print(f"Dropped {n_pre - len(df)} duplicates that varied only in the letter case of the description, n = {len(df)}")
df.drop(columns="Description_CAP_lower", inplace=True)

Dropped 13924 duplicates that varied only in the letter case of the description, n = 446438


In [11]:
# Remove observations with major topic ID 99
n_pre = len(df)
df = df[df["Major_CAP"] != 99]
print(f"Dropped {n_pre - len(df)} observations with major ID = 99")

Dropped 78218 observations with major ID = 99


In [12]:
# Drop columns with more than 10% missing values
# # Retain necessary columns with NAs that might get dropped
# df_na_col_keep = df[['Party_CAP', 'PassH_CAP', 'PassS_CAP', 'DW1']]
n_pre = len(df.columns)
df.dropna(axis=1, thresh=int(len(df) * 0.9), inplace=True)
print(f"Dropped {n_pre - len(df.columns)} columns that contain more than 10% missing values")

# Rearrange columns
df_first_col = ['BillID_corrected', 'Description_CAP', 'Description_CBP', 'Year', 
                 'Major_CAP', 'Major_CBP', 'Party_CAP', 'Party_CBP', 
                 'PassH_CAP', 'PassH_CBP', 'PassS_CAP', 'PassS_CBP',
                 'DW1', 'DW1_impute']
df_remaining_col = [ col for col in df.columns if (col not in df_first_col) ]
df = df[df_first_col + df_remaining_col]

# Display the first few rows of the cleaned dataframe
df.head()

# Save the cleaned data
df_path = os.path.join(data_dir, "bills.csv")
df.to_csv(df_path, index=False)
print(f"Merged and cleaed data from CAP and CBP saved at {df_path}")

Dropped 13 columns that contain more than 10% missing values
Merged and cleaed data from CAP and CBP saved at /Users/haya1/Documents/LanguageModel_Labels-main/congressional_bills/01_cleaning_and_data/bills.csv
